<a href="https://colab.research.google.com/github/AliRaddman/divar-ml-project/blob/main/divar-ml-project/notebooks/work%20/hypothesis_ben.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="text-align: right;">

# آزمون فرض: آیا خانه‌های قدیمی‌ساخت دلبازتر هستند؟

در این سؤال می‌خواهیم این ادعا را بررسی کنیم:

«قدیما خانه‌ها دلبازتر بود.»

برای تبدیل این جمله به یک مسئله آماری، دلبازتر بودن را با متغیر مساحت بنا بررسی می‌کنیم.

متغیر مورد بررسی:

<code>building_size_pos</code>

تعریف گروه‌ها:

<ul>
  <li>خانه قدیمی‌ساخت: <code>construction_year &lt; 1396</code></li>
  <li>خانه جدیدساخت: <code>construction_year &gt;= 1396</code></li>
</ul>

چون سؤال درباره‌ی خانه است، فقط آگهی‌های مسکونی را بررسی می‌کنیم، یعنی دسته‌هایی مثل آپارتمان و خانه/ویلا.  
زمین، مغازه، دفتر کار و املاک صنعتی در این آزمون وارد نمی‌شوند.

در این مرحله فقط داده را آماده و بررسی اولیه می‌کنیم. آزمون فرض در مرحله بعد انجام می‌شود.

</div>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

DATA_PATH = Path("/content/drive/MyDrive/divar_project/cleaned_step3_geo.parquet")

df = pd.read_parquet(DATA_PATH)

print("Data loaded successfully.")
print("Shape:", df.shape)

Mounted at /content/drive
Data loaded successfully.
Shape: (999943, 85)


<div dir="rtl" style="text-align: right;">

## آماده‌سازی دقیق‌تر داده برای آزمون فرض

در خروجی اولیه، مقدارهای بسیار بزرگ و غیرواقعی در ستون متراژ دیده شد.  
برای مثال، مقدار بیشینه‌ی متراژ به ۱۰,۰۰۰,۰۰۰ متر رسیده بود که برای خانه مسکونی منطقی نیست.

از آن‌جایی که این آزمون درباره‌ی میانگین مساحت است، وجود چنین مقدارهایی می‌تواند میانگین را به‌شدت منحرف کند.

بنابراین در این نسخه، علاوه بر انتخاب خانه‌های مسکونی، از flagهای اعتبارسنجی متراژ نیز استفاده می‌کنیم:

<ul>
  <li><code>is_valid_area_basic == True</code></li>
  <li><code>is_valid_area_q01_q99_by_cat3 == True</code></li>
</ul>

این کار کمک می‌کند مقایسه‌ی میانگین مساحت خانه‌های قدیمی و جدید بر اساس داده‌های منطقی‌تر انجام شود.

</div>

In [3]:
residential_cat3 = [
    "apartment-sell",
    "apartment-rent",
    "house-villa-sell",
    "house-villa-rent"
]

hyp_area_df = df[
    (df["cat3_slug"].astype(str).isin(residential_cat3)) &
    (df["construction_year"].notna()) &
    (df["building_size_pos"].notna()) &
    (df["building_size_pos"] > 0) &
    (df["is_valid_area_basic"] == True) &
    (df["is_valid_area_q01_q99_by_cat3"] == True)
].copy()

hyp_area_df["cat3_slug"] = hyp_area_df["cat3_slug"].astype(str)

hyp_area_df["building_age_group"] = np.where(
    hyp_area_df["construction_year"] < 1396,
    "old_before_1396",
    "new_1396_or_after"
)

area_group_summary_clean = (
    hyp_area_df
    .groupby("building_age_group", observed=True)["building_size_pos"]
    .agg(
        ads_count="count",
        mean_area="mean",
        median_area="median",
        std_area="std",
        min_area="min",
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75),
        max_area="max"
    )
    .reset_index()
)

num_cols = ["mean_area", "median_area", "std_area", "min_area", "q1", "q3", "max_area"]
area_group_summary_clean[num_cols] = area_group_summary_clean[num_cols].round(2)

print("Prepared clean hypothesis dataset shape:", hyp_area_df.shape)

print("\nConstruction year range:")
print(hyp_area_df["construction_year"].min(), "to", hyp_area_df["construction_year"].max())

print("\nGroup counts:")
print(hyp_area_df["building_age_group"].value_counts())

print("\nClean area summary by building age group:")
display(area_group_summary_clean)

print("\ncat3_slug distribution in selected clean data:")
display(
    hyp_area_df["cat3_slug"]
    .value_counts()
    .reset_index()
    .rename(columns={"cat3_slug": "cat3_slug", "count": "ads_count"})
)

Prepared clean hypothesis dataset shape: (689093, 86)

Construction year range:
1365.0 to 1403.0

Group counts:
building_age_group
old_before_1396      346748
new_1396_or_after    342345
Name: count, dtype: int64

Clean area summary by building age group:


,building_age_group,ads_count,mean_area,median_area,std_area,min_area,q1,q3,max_area
0,new_1396_or_after,342345,125.18,108.00,103.59,10.00,85.00,142.00,"10,000.00"
1,old_before_1396,346748,107.35,88.00,93.87,10.00,70.00,120.00,"10,000.00"



cat3_slug distribution in selected clean data:


,cat3_slug,ads_count
0,apartment-sell,298489
1,apartment-rent,207773
2,house-villa-sell,119350
3,house-villa-rent,63481


<div dir="rtl" style="text-align: right;">

## تعریف فرض صفر و فرض مقابل

ادعای مورد بررسی این است که میانگین مساحت خانه‌های قدیمی‌ساخت بیشتر از خانه‌های جدیدساخت است.

بنابراین آزمون ما یک آزمون یک‌طرفه است.

فرض صفر:

<code>H0: μ_old ≤ μ_new</code>

فرض مقابل:

<code>H1: μ_old > μ_new</code>

که در آن:

<ul>
  <li><code>μ_old</code>: میانگین مساحت خانه‌های ساخته‌شده قبل از سال ۱۳۹۶</li>
  <li><code>μ_new</code>: میانگین مساحت خانه‌های ساخته‌شده از سال ۱۳۹۶ به بعد</li>
</ul>

برای این مقایسه از آزمون <b>Welch's t-test</b> استفاده می‌کنیم، چون اندازه نمونه‌ها بسیار بزرگ هستند و لازم نیست فرض کنیم واریانس دو گروه دقیقاً برابر است.

</div>

In [4]:
from scipy import stats
import numpy as np

old_area = hyp_area_df.loc[
    hyp_area_df["building_age_group"] == "old_before_1396",
    "building_size_pos"
].dropna()

new_area = hyp_area_df.loc[
    hyp_area_df["building_age_group"] == "new_1396_or_after",
    "building_size_pos"
].dropna()

# Welch one-sided t-test
# H1: mean(old_area) > mean(new_area)
t_stat, p_value = stats.ttest_ind(
    old_area,
    new_area,
    equal_var=False,
    alternative="greater"
)

mean_old = old_area.mean()
mean_new = new_area.mean()
mean_diff = mean_old - mean_new

result_summary = pd.DataFrame({
    "metric": [
        "old_count",
        "new_count",
        "old_mean_area",
        "new_mean_area",
        "mean_difference_old_minus_new",
        "t_statistic",
        "p_value",
        "alpha"
    ],
    "value": [
        len(old_area),
        len(new_area),
        mean_old,
        mean_new,
        mean_diff,
        t_stat,
        p_value,
        0.05
    ]
})

result_summary["value"] = result_summary["value"].astype(float).round(6)

display(result_summary)

if p_value < 0.05:
    print("Decision: Reject H0")
    print("Conclusion: There is statistical evidence that old houses have a larger mean area than new houses.")
else:
    print("Decision: Fail to reject H0")
    print("Conclusion: There is not enough statistical evidence that old houses have a larger mean area than new houses.")

,metric,value
0,old_count,"346,748.00"
1,new_count,"342,345.00"
2,old_mean_area,107.35
3,new_mean_area,125.18
4,mean_difference_old_minus_new,-17.84
5,t_statistic,-74.87
6,p_value,1.00
7,alpha,0.05


Decision: Fail to reject H0
Conclusion: There is not enough statistical evidence that old houses have a larger mean area than new houses.


<div dir="rtl" style="text-align: right;">

## نتیجه نهایی آزمون فرض

در این آزمون بررسی کردیم که آیا میانگین مساحت خانه‌های قدیمی‌ساخت بیشتر از خانه‌های جدیدساخت است یا نه. خانه‌های قدیمی‌ساخت به عنوان خانه‌هایی با سال ساخت قبل از ۱۳۹۶ تعریف شدند و خانه‌های جدیدساخت شامل خانه‌هایی با سال ساخت ۱۳۹۶ و بعد از آن بودند.

فرض صفر و فرض مقابل به صورت زیر تعریف شدند:

<ul>
  <li><code>H0: μ_old ≤ μ_new</code></li>
  <li><code>H1: μ_old > μ_new</code></li>
</ul>

برای انجام آزمون، فقط آگهی‌های مسکونی شامل آپارتمان و خانه/ویلا بررسی شدند. همچنین برای جلوگیری از اثر مقدارهای غیرواقعی، فقط رکوردهایی استفاده شدند که متراژ معتبر داشتند.

بر اساس داده‌های آماده‌شده، تعداد خانه‌های قدیمی‌ساخت برابر با ۳۴۶,۷۴۸ و تعداد خانه‌های جدیدساخت برابر با ۳۴۲,۳۴۵ بود. میانگین مساحت خانه‌های قدیمی‌ساخت حدود ۱۰۷.۳۵ متر مربع و میانگین مساحت خانه‌های جدیدساخت حدود ۱۲۵.۱۸ متر مربع به دست آمد.

اختلاف میانگین به صورت زیر بود:

<code>mean_old - mean_new = -17.84</code>

این مقدار منفی است؛ یعنی در داده‌های بررسی‌شده، میانگین مساحت خانه‌های قدیمی‌ساخت کمتر از خانه‌های جدیدساخت بوده است، نه بیشتر.

نتیجه آزمون Welch's t-test یک‌طرفه نیز مقدارهای زیر را نشان داد:

<ul>
  <li><code>t_statistic = -74.87</code></li>
  <li><code>p_value = 1.00</code></li>
  <li><code>alpha = 0.05</code></li>
</ul>

از آن‌جایی که مقدار <code>p_value</code> از سطح معناداری ۰.۰۵ بزرگ‌تر است، فرض صفر رد نمی‌شود. بنابراین شواهد آماری کافی برای تأیید این ادعا وجود ندارد که خانه‌های قدیمی‌ساخت میانگین مساحت بیشتری نسبت به خانه‌های جدیدساخت دارند.

در واقع، نتیجه توصیفی داده‌ها جهت مخالف این جمله را نشان می‌دهد: در این دیتاست، میانگین مساحت خانه‌های جدیدساخت بیشتر از خانه‌های قدیمی‌ساخت است. بنابراین جمله‌ی «قدیما خانه‌ها دلبازتر بود» با معیار مساحت بنا و بر اساس این داده‌ها تأیید نمی‌شود.

</div>